In [ ]:
pip install catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 30.7 MB/s eta 0:00:00


# Load data

In [ ]:
import pandas as pd

final_df = pd.read_parquet("/content/drive/MyDrive/used_car/final_df.parquet")

print(final_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 383160 entries, 0 to 389308
Data columns (total 23 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   id               383160 non-null  int64  
 1   region           383160 non-null  object 
 2   price            383160 non-null  int64  
 3   manufacturer     383160 non-null  object 
 4   model            383160 non-null  object 
 5   condition        372409 non-null  object 
 6   cylinders        372297 non-null  object 
 7   fuel             380990 non-null  object 
 8   odometer         380952 non-null  float64
 9   title_status     375821 non-null  object 
 10  transmission     381491 non-null  object 
 11  drive            370746 non-null  object 
 12  type             382310 non-null  object 
 13  paint_color      383160 non-null  object 
 14  state            383160 non-null  object 
 15  lat              383160 non-null  float64
 16  long             383160 non-null  float64
 

## Check missing rate function

In [ ]:
def check_missing_rate(df):
    df_null = df.isnull().mean() * 100
    print(f"Length of dataframe: {len(df)}")
    print(df_null[df_null>0].apply(lambda x: f"{(x):.2f}%"))
    del df_null

# Encoding

## Ordinal/One-hot encoding

In [ ]:
import numpy as np
import pandas as pd

encoded_df = final_df.copy()

# Ordinal encoding
condition_mapping = {
    "salvage": 0,
    "fair": 1,
    "good": 2,
    "excellent": 3,
    "like new": 4,
    "new": 5,
}

cylinder_mapping = {
    "other": 0,
    "3 cylinders": 3,
    "4 cylinders": 4,
    "5 cylinders": 5,
    "6 cylinders": 6,
    "8 cylinders": 8,
    "10 cylinders": 10,
    "12 cylinders": 12,
}

encoded_df["condition"] = (
    encoded_df["condition"]
    .map(condition_mapping)
)

encoded_df["cylinders"] = (
    encoded_df["cylinders"]
    .map(cylinder_mapping)
)

# One hot encoding
one_hot_columns = [
    "manufacturer",
    "fuel",
    "title_status",
    "transmission",
    "drive",
    "type",
    "paint_color",
    "state",
    "posting_month",
    "posting_day",
    "posting_hour",
    "posting_weekday",
]

encoded_df = pd.get_dummies(
    encoded_df,
    columns=one_hot_columns,
    dummy_na=True,
    dtype=np.int8,
)
print(encoded_df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 383160 entries, 0 to 389308
Columns: 223 entries, id to posting_weekday_nan
dtypes: float64(7), int64(2), int8(212), object(2)
memory usage: 112.5+ MB
None


In [ ]:
check_missing_rate(encoded_df)

Length of dataframe: 383160
condition         2.81%
cylinders         2.84%
odometer          0.58%
miles_per_year    0.58%
dtype: object


## Filling NAs with mode (categorcal) and median (continuous)

In [ ]:
categorical_cols = ["condition","cylinders"]
continuous_cols = ["odometer"]

for col in categorical_cols:
  encoded_df[col] = encoded_df[col].fillna(
      encoded_df[col].mode()[0]
  )
for col in continuous_cols:
  encoded_df[col] = encoded_df[col].fillna(
      encoded_df[col].median()
  )
encoded_df["miles_per_year"] = encoded_df["odometer"] / (encoded_df["vehicle_age"] + 1)
check_missing_rate(encoded_df)

Length of dataframe: 383160
Series([], dtype: float64)


## Train, validation, test split

In [ ]:
from sklearn.model_selection import train_test_split

model_df = encoded_df.copy()

X = model_df.drop(columns = ["price"])
y = model_df["price"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_test, X_val, y_test, y_val = train_test_split(
    X_eval, y_eval, test_size=0.5, random_state=42
)

# Save the IDs for later
train_ids = X_train["id"].copy()
val_ids = X_val["id"].copy()
test_ids = X_test["id"].copy()

# Remove id before training
X_train = X_train.drop(columns=["id"])
X_val = X_val.drop(columns=["id"])
X_test = X_test.drop(columns=["id"])

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(306528, 221)
(38316, 221)
(38316, 221)


In [ ]:
model_df.to_parquet(
    "/content/drive/MyDrive/used_car/model_df.parquet",
    index=False
)

## Target encoding

In [ ]:
from sklearn.preprocessing import TargetEncoder
import pickle

target_encoder = TargetEncoder(
    target_type = "continuous",
    random_state = 42,
)

target_encoded_cols = ["region","model"]

X_train[target_encoded_cols] = target_encoder.fit_transform(
    X_train[target_encoded_cols],
    y_train,
)

X_val[target_encoded_cols] = target_encoder.transform(
    X_val[target_encoded_cols],
)

X_test[target_encoded_cols] = target_encoder.transform(
    X_test[target_encoded_cols],
)

# Modeling

## Evaluation function (R2, RMSE, MAE)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

def evaluate_regression_model(y, pred):
    mae = mean_absolute_error(y, pred)
    rmse = np.sqrt(mean_squared_error(y, pred))
    r2 = r2_score(y, pred)

    print(f"MAE   : ${mae}")
    print(f"RMSE  : ${rmse}")
    print(f"R2    : {r2}")

    return mae, rmse, r2

## Baseline models

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),
    "Random Forest": RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
    ),
    "XGBoost": XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        tree_method="hist",
        device="cuda"
    ),
    "LightGBM": LGBMRegressor(
        objective="regression",
        random_state=42,
        n_jobs=-1,
        device="gpu",
        verbosity=0
    ),
    "CatBoost": CatBoostRegressor(
        verbose=False,
        random_seed=42,
        task_type="GPU",
        ),
}

results = []

for name, model in models.items():
    print(f"\n Running model: {name}")
    model.fit(X_train, y_train)

    pred = model.predict(X_val)
    mae, rmse, r2 = evaluate_regression_model(y_val, pred)
    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    })


results_df = pd.DataFrame(results).sort_values(by="R2", ascending=False)
results_df = results_df.set_index("Model")
print(results_df)



 Running model: Linear Regression
MAE   : $6939.17
RMSE  : $11149.26
R2    : 0.49

 Running model: Ridge


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=7.3601e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)


MAE   : $6939.21
RMSE  : $11149.13
R2    : 0.49

 Running model: Lasso
MAE   : $6937.85
RMSE  : $11149.30
R2    : 0.49

 Running model: Decision Tree
MAE   : $2732.07
RMSE  : $8164.72
R2    : 0.72

 Running model: Random Forest
MAE   : $2181.85
RMSE  : $6412.16
R2    : 0.83

 Running model: XGBoost
MAE   : $3539.92
RMSE  : $7262.31
R2    : 0.78

 Running model: LightGBM
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
MAE   : $3888.22
RMSE  : $7663.22
R2    : 0.76

 Running model: CatBoost
MAE   : $3621.79
RMSE  : $7370.82
R2    : 0.78
                           MAE          RMSE        R2
Model                                                 
Random Forest      2181.845040   6412.157996  0.830355
XGBoost            3539.923096   7262.313681  0.782388
CatBoost           3621.788120   7370.823082  0.775836
LightGBM           3888.216319   7663.222569  0.757699
Decision Tree      2732.068579   8164.723920  0.724947
Ridge              6939.205999  11149.1341

## Hyperparameter tuning on XGBoost (Optuna)

In [ ]:
import optuna

def objective(trial):

    params = {
        "objective": "reg:squarederror",
        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.005,
            0.2,
            log=True,
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            4,
            12,
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            12,
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5,
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0,
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0,
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0,
            10,
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.1,
            20,
            log=True,
        ),

        "n_estimators": trial.suggest_int(
            "n_estimators",
            2000,
            10000
        ),
        "tree_method": "hist",
        "device": "cuda",
        "random_state": 42,
        "early_stopping_rounds": 100,
    }

    model = XGBRegressor(**params)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    pred = model.predict(X_val)

    rmse = np.sqrt(
        mean_squared_error(
            y_val,
            pred
        )
    )

    trial.set_user_attr(
        "best_iteration",
        model.best_iteration
    )

    return rmse

In [ ]:
study = optuna.create_study(
    direction="minimize",
    study_name="xgb_used_car"
)

study.optimize(
    objective,
    n_trials=200,
    show_progress_bar=True,
)

print("Best RMSE:")
print(study.best_value)

print("Best Parameters:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

print("Best Iteration:")
print(
    study.best_trial.user_attrs["best_iteration"]
)

[I 2026-08-08 19:12:43,325] A new study created in memory with name: xgb_used_car


  0%|          | 0/200 [00:00<?, ?it/s]

[I 2026-08-08 19:13:00,613] Trial 0 finished with value: 6139.4791310012615 and parameters: {'learning_rate': 0.06110936289274055, 'max_depth': 9, 'min_child_weight': 12, 'gamma': 2.767880494007252, 'subsample': 0.6598901537339309, 'colsample_bytree': 0.9067527889930975, 'reg_alpha': 0.09102806584858292, 'reg_lambda': 14.017083756559638, 'n_estimators': 2394}. Best is trial 0 with value: 6139.4791310012615.
[I 2026-08-08 19:13:36,698] Trial 1 finished with value: 6330.014849903592 and parameters: {'learning_rate': 0.01083036861580939, 'max_depth': 9, 'min_child_weight': 6, 'gamma': 2.8739272083213976, 'subsample': 0.8625500619938461, 'colsample_bytree': 0.7358968087797527, 'reg_alpha': 4.365908778384532, 'reg_lambda': 11.992805180867961, 'n_estimators': 4931}. Best is trial 0 with value: 6139.4791310012615.
[I 2026-08-08 19:14:00,473] Trial 2 finished with value: 6322.114519684059 and parameters: {'learning_rate': 0.11333357544932023, 'max_depth': 5, 'min_child_weight': 6, 'gamma': 4.9

## Tuned final model

In [ ]:
from xgboost import XGBRegressor
# best_params = study.best_params.copy()

best_params = {
    "learning_rate": 0.032391806082571674,
    "max_depth": 12,
    "min_child_weight": 4,
    "gamma": 1.3122052468350847,
    "subsample": 0.8504168706670547,
    "colsample_bytree": 0.66691885862711,
    "reg_alpha": 8.463875760346582,
    "reg_lambda": 2.7629507761330325,
    "n_estimators": 9769,
}
# best_iteration = 9765

xgb_model = XGBRegressor(
    objective = "reg:squarederror",
    tree_method = "hist",
    device= "cuda",
    random_state = 42,
    early_stopping_rounds = 100,
    **best_params,
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

print("Validation results")
pred = xgb_model.predict(X_val)
mae, rmse, r2 = evaluate_regression_model(y_val, pred)

print("Test results")
pred = xgb_model.predict(X_test)
mae, rmse, r2 = evaluate_regression_model(y_test, pred)

Validation results


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [04:31:20] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


MAE   : $1898.1407470703125
RMSE  : $5729.844500507846
R2    : 0.8645375967025757
Test results
MAE   : $1901.459228515625
RMSE  : $4579.217182008296
R2    : 0.909377932548523


In [ ]:
X_train_final = pd.concat([X_train, X_val], axis=0)
y_train_final = pd.concat([y_train, y_val], axis=0)

model = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    device="cuda",
    random_state=42,
    **best_params,
)

model.fit(
    X_train_final,
    y_train_final,
    verbose=False,
)

print("Test results (retrained on train+val)")
pred = model.predict(X_test)
mae, rmse, r2 = evaluate_regression_model(y_test, pred)

Test results (retrained on train+val)
MAE   : $1846.2786865234375
RMSE  : $4541.018167768105
R2    : 0.9108835458755493


In [ ]:
predicted_df = pd.DataFrame({
    "id": test_ids.values,
    "predicted_price": pred,
})
predicted_df.to_parquet("/content/drive/MyDrive/used_car/predicted_df.parquet")

## SHAP

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test, check_additivity=False)

# Convert to DataFrame
shap_df = pd.DataFrame(
    shap_values,
    columns=X_test.columns,
)

# Attach the original listing IDs
shap_df["id"] = test_ids.values

# Export
shap_df.to_parquet(
    "/content/drive/MyDrive/used_car/shap_values.parquet",
    index=False,
)

print(shap_df.head())

       region        model   condition    cylinders     odometer         lat  \
0  -33.501968 -2574.205078 -234.443008 -1124.522583 -4592.113770  160.348892   
1  103.799927 -2209.158691 -102.762978   -98.412880 -3762.501953  -49.950417   
2   24.817827 -4994.106934  292.136841   188.435913 -1814.916260  134.719406   
3  115.604622   374.015991  633.197144  -203.931335 -2654.845703  156.346283   
4   61.526619  -745.085327 -221.824036  2319.745850 -5225.122559   38.536510   

          long  vehicle_age  miles_per_year  manufacturer_acura  ...  \
0 -1235.319946 -3953.635742      144.213638          -11.662572  ...   
1   418.192810 -1668.335083      594.228577           -3.014570  ...   
2  -234.036652  -403.404999      429.080383          -13.004307  ...   
3    98.988144 -8121.052734    -1177.297119            9.575929  ...   
4    -6.292630 -6886.335449      -77.555275            4.615090  ...   

   posting_hour_nan  posting_weekday_0.0  posting_weekday_1.0  \
0               0.0  